# 03 — Attention bottleneck visualizer

Visualizzazione della mask 4D che realizza il true bottleneck (`src/bottleneck.py`): matrice causale vs bottleneck, archi rimossi, gestione del padding. Le celle finali (opzionali, `RUN_QWEN=True`) replicano su Qwen reale i controlli leak della suite integration.

Invariante: dopo `[COMPRESS]` (posizione c), nessuna query puo' leggere chiavi prima di c — l'unica rotta verso il contesto e' l'hidden state dell'anchor.

In [ ]:
# Provenance header: every figure must be traceable to the state that made it.
import json, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

def sh(*cmd):
    try:
        return subprocess.check_output(cmd, cwd=ROOT, text=True).strip()
    except Exception as e:
        return f"<unavailable: {e}>"

GIT_COMMIT = sh("git", "rev-parse", "--short", "HEAD")
GIT_DIRTY = bool(sh("git", "status", "--porcelain"))
MANIFEST_PATH = ROOT / "data/processed/manifest.json"
MANIFEST = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
print(f"commit={GIT_COMMIT} dirty={GIT_DIRTY}")
print(f"manifest: {'present, created ' + MANIFEST['created_at'] if MANIFEST else 'ABSENT'}")


In [ ]:
import torch
from src.bottleneck import build_bottleneck_mask, build_causal_mask

# Layout di esempio: 4 ctx + [COMPRESS] + 3 filler + [RECALL] + 3 target = 12
SEQ, C = 12, 4
R = 8
labels = [f"ctx{i}" for i in range(4)] + ["[CMP]"] + \
         [f"fil{i}" for i in range(3)] + ["[RCL]"] + [f"tgt{i}" for i in range(3)]
attn2d = torch.ones(1, SEQ)
bott = build_bottleneck_mask(attn2d, torch.tensor([C]))
caus = build_causal_mask(attn2d)
allowed_b = (bott[0, 0] == 0)
allowed_c = (caus[0, 0] == 0)
print(f"archi permessi: causale={int(allowed_c.sum())}, "
      f"bottleneck={int(allowed_b.sum())}, rimossi={int((allowed_c & ~allowed_b).sum())}")


In [ ]:
# Heatmap due-toni etichettate (mai colore da solo: colorbar con tick testuali)
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def show_mask(ax, allowed, title):
    cmap = ListedColormap(["#e8e7e2", "#2a78d6"])   # blocked, allowed
    im = ax.imshow(allowed.int(), cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(SEQ)); ax.set_yticks(range(SEQ))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("key"); ax.set_ylabel("query")
    ax.axvline(C - 0.5, color="#0b0b0b", lw=0.8, ls="--")
    ax.axhline(C - 0.5, color="#0b0b0b", lw=0.8, ls="--")
    ax.set_title(title, fontsize=10)
    return im

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_mask(axes[0], allowed_c, "causale ordinaria (v0)")
show_mask(axes[1], allowed_b, "bottleneck (v2)")
removed = (allowed_c & ~allowed_b)
im = axes[2].imshow(removed.int(), cmap=ListedColormap(["#e8e7e2", "#e34948"]),
                    vmin=0, vmax=1)
axes[2].set_xticks(range(SEQ)); axes[2].set_yticks(range(SEQ))
axes[2].set_xticklabels(labels, rotation=90, fontsize=7)
axes[2].set_yticklabels(labels, fontsize=7)
axes[2].set_title("archi RIMOSSI dal bottleneck", fontsize=10)
cbar = fig.colorbar(im, ax=axes, shrink=0.6, ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(["blocked/assente", "allowed/rimosso"])
fig.savefig(ROOT / "results/figures/bottleneck_mask.png", dpi=150, bbox_inches="tight")
plt.show()
# Lettura: nel pannello centrale ogni query dopo [CMP] (righe 5+) ha accesso
# SOLO alle chiavi da [CMP] in poi; il pannello destro mostra che gli archi
# rimossi sono esattamente query-post-anchor -> chiavi-di-contesto.


In [ ]:
# Right padding: le chiavi di padding sono sempre bloccate; il LEFT padding
# viene rifiutato (righe di query interamente mascherate -> NaN poisoning)
attn_pad = torch.tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])
m = build_bottleneck_mask(attn_pad, torch.tensor([C]))
print("chiavi padding bloccate per ogni query:",
      bool((m[0, 0, :, 8:] != 0).all()))
try:
    build_bottleneck_mask(torch.tensor([[0, 0, 1, 1, 1, 1]]), torch.tensor([3]))
except Exception as e:
    print("left padding rifiutato:", type(e).__name__, "-", str(e)[:80])


In [ ]:
# Verifica numerica della truth table (stessa logica del test unitario)
ok = True
for q in range(SEQ):
    for k in range(SEQ):
        expected = (k <= q) if q <= C else (C <= k <= q)
        ok &= (bool(allowed_b[q, k]) == expected)
print("truth table esaustiva:", "OK" if ok else "VIOLATA")


## Celle opzionali — Qwen reale

Richiedono il checkpoint `Qwen/Qwen2.5-0.5B` in cache HF (~1 GB RAM). Replicano i controlli chiave di `tests/test_qwen_integration.py`: la mask e' onorata e non c'e' leak attorno all'anchor.

In [ ]:
RUN_QWEN = False   # metti True per eseguire (~30s su CPU)
if RUN_QWEN:
    from src.bottleneck import validate_layout
    from src.config import COMPRESS_TOKEN, RECALL_TOKEN, TrainConfig
    from src.model import load_base_model, load_tokenizer

    cfg = TrainConfig()
    tok = load_tokenizer(cfg.model_name)
    model = load_base_model(cfg.model_name, tok, dtype=torch.float32).eval()
    prompt = ("The secret launch code is 7319 and the probe departs from "
              f"Cape Canaveral.\n{COMPRESS_TOKEN}\nfiller words\n{RECALL_TOKEN}\n")
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"]
    c, _ = validate_layout(ids, tok, require_recall=False)
    a2d = torch.ones_like(ids)
    with torch.no_grad():
        ref = model(input_ids=ids, attention_mask=a2d, use_cache=False).logits
        ours = model(input_ids=ids, attention_mask=build_causal_mask(a2d),
                     use_cache=False).logits
        bt = model(input_ids=ids, attention_mask=build_bottleneck_mask(a2d, c),
                   use_cache=False).logits
    print("mask 4D onorata (causale 4D == default):",
          torch.allclose(ref, ours, atol=1e-4))
    print("bottleneck rimuove davvero archi (logits diversi):",
          not torch.allclose(ref[0, -1], bt[0, -1], atol=1e-4))
